# Tutorial 101: Group Sequential Testing in EarlySign

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
!pip install "earlysign" "ibis-framework[duckdb]"


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [19]:
import pandas as pd

data = pd.DataFrame(
    {
        "action_date": ["2025-10-14", "2025-10-15", "2025-10-16"],
        "A_count": [100, 100, 100],
        "B_count": [100, 100, 100],
        "A_sum": [1, 2, 1],
        "B_sum": [3, 5, 7],
    }
)
data

,action_date,A_count,B_count,A_sum,B_sum
0,2025-10-14,100,100,1,3
1,2025-10-15,100,100,2,5
2,2025-10-16,100,100,1,7


In [20]:
import ibis
from earlysign.api.ab_tests import BinomialABTest

conn = ibis.connect("duckdb://:memory:")

test = BinomialABTest(conn, "my_fantastic_experiment")

## We need to select alpha, power, max sample size, etc.
# designer = test.designer()
# designer.select()
# design.to_json()

# Use the designer to generate a valid design object
# designer = test.designer()

# Select design parameters (alpha, power, effect_size, etc.)
# max_n (maximum sample size) can be specified as max_n or similar depending on the API.
design = dict(
    alpha=0.05,
    power=0.8,
    effect_size=0.01,
    tails=2,
    scale="z",
    efficacy=dict(style="alpha_spending", family="obf"),
    futility=dict(mode="symmetric"),
    max_n=300,  # max_n: maximum total sample size across both groups
)
# Optionally, inspect or serialize the design
# print(design)
# design.to_json()
test.set_design(design)

# test.plot_design()

In [21]:
from earlysign.core.ledger import Ledger

Ledger(conn, "my_fantastic_experiment").t.select(
    "payload_type", "payload", "labels"
).execute()

,payload_type,payload,labels
0,GroupSequential/Design,"{'alpha': 0.05, 'effect_size': 0.01, 'efficacy...","{'experiment_id': 'my_fantastic_experiment', '..."


### Design Phase
Let's first make some choices on the design of the experiment.
Toward the end, we will select the tentative sample size based on some belief on the effect size.

In [ ]:
## Let's conduct the experiment! (Here, we are only simulating it in retrospect)
## Experiment loop
for row in data.sort_values("action_date", ascending=True).to_dict(orient="records"):
    ## Re-instantiate test object
    test = BinomialABTest(conn, "my_fantastic_experiment")

    ## Observe data and run updates
    test.update(dict(
        nA=row["A_count"], nB=row["B_count"], mA=row["A_sum"], mB=row["B_sum"]
    ))

    ## Check updated test status
    status = test.status()
    print(status)
    if status.stop_recommended:
        break  # This test template only has recommendations for stopping. We optionally follow it to enjoy early stopping.

    ## The process may well be reset every day
    del test

    # break  # For debug

## Re-instantiate test object
test = BinomialABTest(conn, "my_fantastic_experiment")
## Check out the reports
# test.report_results()  # Check final result
# test.report_history()  # Visualize test history

State(stop_recommended=False)
State(stop_recommended=False)
State(stop_recommended=False)


In [ ]:
## Comparison: Single-shot test